# Taller 4. GLM-OCR: layout PP-DocLayoutV3 + reconocimiento GLM-OCR

**Ejercicio avanzado.** El pipeline oficial de GLM-OCR segmenta la página con **PP-DocLayoutV3** y luego envía cada región al modelo **GLM-OCR (0,9 B parámetros)**. El modelo corre en un servidor local de **Ollama** dentro del mismo entorno: no se necesita API comercial ni clave.

**Qué produce:** la visualización del layout que genera el SDK, la transcripción en Markdown, el JSON con las regiones en orden de lectura y los recortes.

**Entorno:** Colab con GPU T4. La instalación descarga Ollama y unos 2,7 GB del modelo. En CPU funciona, pero cada región tarda varios segundos.

## 0. Instalar Ollama, el modelo y el SDK

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "glmocr[selfhosted]==0.1.5", "pypdfium2"], check=True)
    print("Instalados Ollama y glmocr 0.1.5")
else:
    print("Fuera de Colab no se instala nada: preparar el entorno según cuadernos/README.md.")

## 1. Iniciar el servidor del modelo

Si Ollama ya está corriendo (por ejemplo, en una computadora propia), se reutiliza. La primera vez se descarga el modelo `glm-ocr`.

In [ ]:
import subprocess
import time

import requests

OLLAMA = "http://localhost:11434"
MODELO_OLLAMA = "glm-ocr:latest"


def ollama_activo():
    try:
        return requests.get(f"{OLLAMA}/api/version", timeout=3).ok
    except requests.RequestException:
        return False


if not ollama_activo():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_activo():
            break
        time.sleep(2)
    else:
        raise RuntimeError("Ollama no respondió; revisar ollama.log")
subprocess.run(["ollama", "pull", MODELO_OLLAMA], check=True, stdout=subprocess.DEVNULL)
# Cargar el modelo en memoria antes de usarlo: la primera carga puede superar el tiempo de espera del SDK.
requests.post(f"{OLLAMA}/api/generate", json={"model": MODELO_OLLAMA, "keep_alive": "60m"}, timeout=600).raise_for_status()
print("Ollama", requests.get(f"{OLLAMA}/api/version").json()["version"], "con", MODELO_OLLAMA, "cargado")

## 2. Elegir las imágenes

Por defecto se procesa la **página común del taller** (*Centro Gallego*, Montevideo, 1917): la misma página en los cuatro cuadernos permite comparar modelos. Cambiá `SELECCION` para trabajar con otras fuentes:

| Valor | Qué procesa |
|---|---|
| `"comun"` | la página común (recomendado para la primera pasada) |
| `"prensa"` | todas las páginas de prensa del taller |
| `"manuscritos"` | cartas, actas y otros manuscritos del taller |
| `"todas"` | prensa y manuscritos |
| `"propias"` | tus imágenes o PDF; en Colab aparecen dos botones para subir **varios archivos** o **una carpeta completa**; fuera de Colab se leen de `mis_imagenes/` (incluidas sus subcarpetas) |

Cada página de un PDF se convierte en una imagen PNG (resolución `DPI_PDF = 200`) y se procesa como una página más.

**No subas material sensible o con restricciones de uso** a servicios externos.

In [ ]:
SELECCION = "comun"

In [ ]:
import csv
import sys
import urllib.request
import uuid
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from PIL import Image

EN_COLAB = "google.colab" in sys.modules
REPO_RAW = "https://raw.githubusercontent.com/agusnieto77/sicss-uruguay-2026/main/cuadernos/imagenes/"
BASE = Path("/content/taller") if EN_COLAB else Path.cwd()
CARPETA_IMAGENES = BASE / "imagenes"
EXTENSIONES_IMAGEN = {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}
EXTENSIONES = EXTENSIONES_IMAGEN | {".pdf"}
DPI_PDF = 200  # resolución con la que cada página de un PDF se convierte en imagen


def traer(relativa):
    """Ruta local de un archivo del taller; en Colab lo descarga del repositorio la primera vez."""
    destino = CARPETA_IMAGENES / relativa
    if EN_COLAB and not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + relativa, destino)
    if not destino.exists():
        raise FileNotFoundError(f"No encuentro {destino}. Abrí el cuaderno desde la carpeta cuadernos/ del repositorio.")
    return destino


def mostrar(imagen, ancho=1000):
    """Muestra una imagen PIL reducida a un ancho cómodo para la pantalla."""
    if imagen.width > ancho:
        imagen = imagen.resize((ancho, round(imagen.height * ancho / imagen.width)))
    display(imagen)


def admitido(nombre):
    """Imágenes y PDF, sin archivos ocultos (por ejemplo, los `._foto.jpg` que deja macOS en las carpetas)."""
    return not Path(nombre).name.startswith(".") and Path(nombre).suffix.lower() in EXTENSIONES


def pdf_a_imagenes(pdf, carpeta):
    """Convierte cada página de un PDF en una imagen PNG y devuelve sus rutas."""
    import pypdfium2 as pdfium

    carpeta.mkdir(parents=True, exist_ok=True)
    documento = pdfium.PdfDocument(pdf)
    paginas = []
    for indice in range(len(documento)):
        pagina = documento[indice]
        destino = carpeta / f"{pdf.stem}_p{indice + 1:03d}.png"
        pagina.render(scale=DPI_PDF / 72).to_pil().convert("RGB").save(destino)
        pagina.close()
        paginas.append(destino)
    documento.close()
    return paginas


SELECTOR_COLAB = """
<div style="display:flex; gap:2em; align-items:center; flex-wrap:wrap">
  <label><b>Elegir archivos</b> (imágenes y PDF):
    <input type="file" id="__ID__-archivos" multiple accept="image/*,.tif,.tiff,.pdf"></label>
  <span><b>o una carpeta completa</b>:
    <input type="file" id="__ID__-carpeta" webkitdirectory style="display:none"
      onchange="document.getElementById('__ID__-estado').textContent = this.files.length
        ? this.files[0].webkitRelativePath.split('/')[0] + ' (' + this.files.length + ' archivos)' : ''">
    <button type="button" onclick="document.getElementById('__ID__-carpeta').click()">Seleccionar carpeta</button>
    <span id="__ID__-estado">Ninguna carpeta seleccionada</span></span>
</div>
<script>
window.subidaTaller = {
  elegir: (id) => new Promise((resolve) => {
    for (const sufijo of ["archivos", "carpeta"]) {
      document.getElementById(id + "-" + sufijo).addEventListener("change", (evento) => {
        window.subidaTaller.lista = Array.from(evento.target.files);
        resolve(window.subidaTaller.lista.map((f) => f.webkitRelativePath || f.name));
      });
    }
  }),
  trozo: (indice, inicio, fin) => new Promise((resolve, reject) => {
    const lector = new FileReader();
    lector.onload = () => resolve(lector.result.split(",")[1] || "");
    lector.onerror = () => reject(lector.error);
    lector.readAsDataURL(window.subidaTaller.lista[indice].slice(inicio, fin));
  }),
  tamano: (indice) => window.subidaTaller.lista[indice].size,
};
</script>
"""


def subir_en_colab(carpeta, trozo=1024 * 1024):
    """Diálogo de Colab para elegir varios archivos o una carpeta; guarda imágenes y PDF en `carpeta`."""
    import base64

    from google.colab import output

    identificador = "subida-" + uuid.uuid4().hex
    display(HTML(SELECTOR_COLAB.replace("__ID__", identificador)))
    nombres = output.eval_js(f'window.subidaTaller.elegir("{identificador}")')
    for indice, nombre in enumerate(nombres):
        if not admitido(nombre):
            print("Ignorado (no es imagen ni PDF):", nombre)
            continue
        destino = carpeta.joinpath(*Path(nombre).parts)  # una carpeta conserva sus subcarpetas
        destino.parent.mkdir(parents=True, exist_ok=True)
        tamano = output.eval_js(f"window.subidaTaller.tamano({indice})")
        with open(destino, "wb") as archivo:
            for inicio in range(0, tamano, trozo):
                datos = output.eval_js(f"window.subidaTaller.trozo({indice}, {inicio}, {inicio + trozo})")
                archivo.write(base64.b64decode(datos))
        print(f"Subido: {nombre} ({tamano / 1e6:.1f} MB)")


FICHAS = {}
if SELECCION == "propias":
    carpeta_propia = BASE / "mis_imagenes"
    carpeta_propia.mkdir(parents=True, exist_ok=True)
    if EN_COLAB:
        subir_en_colab(carpeta_propia)
    IMAGENES = []
    for archivo in sorted(p for p in carpeta_propia.rglob("*") if p.is_file() and admitido(p.name)):
        if archivo.suffix.lower() == ".pdf":
            paginas = pdf_a_imagenes(archivo, BASE / "paginas_pdf")
            print(f"{archivo.name}: {len(paginas)} página(s) convertidas a imagen a {DPI_PDF} dpi")
            IMAGENES += paginas
        else:
            IMAGENES.append(archivo)
else:
    with open(traer("fuentes.csv"), encoding="utf-8-sig") as archivo:
        fichas = list(csv.DictReader(archivo, delimiter=";"))
    filtros = {
        "comun": lambda f: f["pagina_comun"] == "si",
        "prensa": lambda f: f["tipo"] == "prensa",
        "manuscritos": lambda f: f["tipo"] != "prensa",
        "todas": lambda f: True,
    }
    if SELECCION not in filtros:
        raise ValueError(f"SELECCION debe ser uno de {sorted(filtros) + ['propias']}")
    elegidas = [f for f in fichas if filtros[SELECCION](f)]
    IMAGENES = [traer(f["archivo"]) for f in elegidas]
    FICHAS = {ruta.name: ficha for ruta, ficha in zip(IMAGENES, elegidas)}
    display(pd.DataFrame(elegidas)[["archivo", "tipo", "titulo", "fecha", "lugar", "institucion", "licencia"]])

assert IMAGENES, "No hay imágenes para procesar: revisá SELECCION o la carpeta mis_imagenes/."
print(f"{len(IMAGENES)} imagen(es):", [p.name for p in IMAGENES])

## 3. Configurar el pipeline

La configuración parte de la que trae el SDK y cambia solo la conexión con el servidor del modelo.

Por defecto, GLM-OCR **descarta** las regiones que su detector clasifica como encabezado, pie de página, número de página, nota al pie o texto marginal. `CONSERVAR_ENCABEZADOS = True` hace que también se transcriban: compará ambas salidas.

In [ ]:
import importlib.resources

import torch
import yaml

CONSERVAR_ENCABEZADOS = False
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
SALIDA = BASE / "resultados" / "04_glm_ocr"
SALIDA.mkdir(parents=True, exist_ok=True)

config = yaml.safe_load((importlib.resources.files("glmocr") / "config.yaml").read_text(encoding="utf-8"))
config["pipeline"]["maas"]["enabled"] = False
config["pipeline"]["ocr_api"].update({
    "api_host": "localhost",
    "api_port": 11434,
    "api_path": "/api/generate",
    "api_mode": "ollama_generate",
    "model": MODELO_OLLAMA,
    "connect_timeout": 180,
    "request_timeout": 600,
})
config["pipeline"]["max_workers"] = 4
mapa = config["pipeline"]["layout"]["label_task_mapping"]
if CONSERVAR_ENCABEZADOS:
    rescatadas = [e for e in ("header", "footer", "number", "footnote", "aside_text") if e in mapa["abandon"]]
    mapa["abandon"] = [e for e in mapa["abandon"] if e not in rescatadas]
    mapa["text"] = mapa["text"] + rescatadas
CONFIG = SALIDA / "config_glmocr.yaml"
CONFIG.write_text(yaml.safe_dump(config, allow_unicode=True), encoding="utf-8")
print("Se descartan sin transcribir:", mapa["abandon"])

## 4. Procesar las páginas

In [ ]:
import json

from glmocr import GlmOcr

regiones = []
with GlmOcr(config_path=str(CONFIG), layout_device=DISPOSITIVO) as parser:
    for pagina in IMAGENES:
        resultado = parser.parse(str(pagina))
        resultado.save(output_dir=str(SALIDA))
        destino = SALIDA / pagina.stem
        print("Página:", pagina.name)
        for vista in sorted((destino / "layout_vis").glob("*")):
            mostrar(Image.open(vista))
        datos = resultado.json_result
        if isinstance(datos, str):
            datos = json.loads(datos)
        ancho, alto = Image.open(pagina).size
        if not any(datos):
            print(f"ATENCIÓN: {pagina.name} no produjo regiones. Revisar los mensajes de error de arriba "
                  "(por ejemplo, memoria de GPU insuficiente o servidor de Ollama sin responder).")
        for numero_pagina, bloques in enumerate(datos):
            for bloque in bloques:
                x1, y1, x2, y2 = bloque["bbox_2d"]  # coordenadas normalizadas de 0 a 1000
                regiones.append({
                    "pagina": pagina.name,
                    "orden": bloque["index"] + 1,
                    "clase": bloque.get("native_label", bloque["label"]),
                    "x1": round(x1 * ancho / 1000), "y1": round(y1 * alto / 1000),
                    "x2": round(x2 * ancho / 1000), "y2": round(y2 * alto / 1000),
                    "texto": bloque.get("content") or "",
                    "evaluacion_humana": "",
                })
        print("TRANSCRIPCIÓN (primeros 4000 caracteres)\n", str(resultado.markdown_result)[:4000])

regiones = pd.DataFrame(regiones, columns=["pagina", "orden", "clase", "x1", "y1", "x2", "y2", "texto", "evaluacion_humana"])
regiones.to_csv(SALIDA / "bloques.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
print("Guardado en:", SALIDA)

## 5. Regiones en orden de lectura

In [ ]:
pd.set_option("display.max_colwidth", 120)
display(regiones[["pagina", "orden", "clase", "texto"]])

## 6. Discusión

1. ¿Qué regiones no aparecen en la transcripción? Volvé a ejecutar con `CONSERVAR_ENCABEZADOS = True` y compará.
2. ¿Aparecen caracteres de otros alfabetos dentro de nombres propios o palabras en español? ¿Por qué un modelo multilingüe produce ese tipo de error?
3. Compará la misma región con los Talleres 2 y 3: ¿qué modelo altera menos los nombres, fechas y cifras?
4. ¿Qué decisiones del pipeline (detector, clases descartadas, orden) quedaron fuera del control del investigador?

## Registro de mediaciones

Toda transcripción automática depende de la herramienta, la versión, la configuración y el equipo que la produjeron. Esta celda guarda esos datos en `procesamiento.json`, junto a los resultados, para poder citarlos y reproducirlos.

In [ ]:
import datetime
import importlib.metadata as metadata
import json
import platform

registro = {
    "herramienta": "GLM-OCR (SDK glmocr + Ollama)",
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "entorno": "Google Colab" if EN_COLAB else platform.platform(),
    "python": platform.python_version(),
    "dispositivo": DISPOSITIVO,
    "imagenes": [p.name for p in IMAGENES],
    "versiones": {"glmocr": metadata.version("glmocr"), "ollama": requests.get(f"{OLLAMA}/api/version").json()["version"]},
    "modelo": {"layout": config["pipeline"]["layout"]["model_dir"], "ocr": MODELO_OLLAMA},
    "parametros": {"conservar_encabezados": CONSERVAR_ENCABEZADOS, "descartadas": mapa["abandon"], "umbral_layout": config["pipeline"]["layout"]["threshold"]},
}
(SALIDA / "procesamiento.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(registro, ensure_ascii=False, indent=2))

## Descargar resultados

In [ ]:
import shutil

if EN_COLAB:
    from google.colab import files

    files.download(shutil.make_archive(f"/content/resultados_04_glm_ocr", "zip", SALIDA))
else:
    print("Resultados en:", SALIDA.resolve())